In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [4]:
first_rdd = spark.sparkContext.parallelize([(1, 2), (3, 4), (3, 6), (3, 6), (1, 3)])
second_rdd = spark.sparkContext.parallelize([(3, 9), (2, 3)])


## Inner Join

In [5]:
first_rdd.join(second_rdd).collect()

[(3, (4, 9)), (3, (6, 9)), (3, (6, 9))]

## Left Outer Join

In [6]:
first_rdd.leftOuterJoin(second_rdd).collect()

[(1, (2, None)), (1, (3, None)), (3, (4, 9)), (3, (6, 9)), (3, (6, 9))]

## Right Outer Join

In [7]:
first_rdd.rightOuterJoin(second_rdd).collect()

[(2, (None, 3)), (3, (4, 9)), (3, (6, 9)), (3, (6, 9))]

## Join data from 2 files

In [10]:
weather_rdd = spark.sparkContext.textFile('/content/weather.csv')
city_rdd = spark.sparkContext.textFile('/content/zips_city.csv')

In [11]:
weather_rdd.take(5)

['2016-05-09,234893,34',
 '2019-09-08,234896,3',
 '2019-11-19,234895,24',
 '2017-04-04,234900,43',
 '2013-12-04,234900,47']

In [12]:
city_rdd.take(5)

['234890,London',
 '234891,Dublin',
 '234892,Paris',
 '234893,New York',
 '234894,Washington']

In [20]:
processed_weather_rdd = weather_rdd.map(lambda x: (x.split(',')[1], int(x.split(',')[2]))).reduceByKey(lambda x, y: x if x > y else y)
processed_weather_rdd.take(5)

[('234896', 40),
 ('234895', 45),
 ('234897', 47),
 ('234890', 41),
 ('234898', 47)]

In [21]:
processed_city_rdd = city_rdd.map(lambda x: (x.split(',')[0], x.split(',')[1]))
processed_city_rdd.take(5)

[('234890', 'London'),
 ('234891', 'Dublin'),
 ('234892', 'Paris'),
 ('234893', 'New York'),
 ('234894', 'Washington')]

In [28]:
processed_weather_rdd.join(processed_city_rdd).map(lambda x: x[1]).collect()

[(41, 'London'),
 (47, 'Tblisi'),
 (41, 'New York'),
 (40, 'Tokyo'),
 (45, 'New Delhi'),
 (47, 'Riyadh'),
 (31, 'Dublin'),
 (47, 'Berlin'),
 (40, 'Washington'),
 (44, 'Kyiv'),
 (40, 'Paris')]

## Notes:
- Always try to do data cleanup, filtering or local aggregation before join.
- Join is expensive operation, so avoid it if possible.
- Broadcast join is a good option if one of the datasets is small enough to fit in memory.
